# Gerador de Dados Sintéticos com IA

Notebook interativo para gerar datasets sintéticos usando modelos de IA, com interface Gradio para configuração e download.

## Funcionalidades

- **Múltiplos provedores:** OpenAI (GPT-4o-mini, GPT-4o) e HuggingFace (Llama 3.1, Mistral 7B, Qwen 2.5)
- **Templates pré-definidos:** Perfis de Funcionários, Vendas de Sorvete, Avaliações de Produtos, Pacientes de Clínica
- **Modo personalizado:** descreva livremente o dataset que deseja gerar
- **Exportação em CSV** com encoding compatível com Excel (UTF-8 BOM)
- **Parsing robusto** com 3 estratégias de fallback para extrair JSON da resposta do modelo

## Pré-requisitos

- `OPENAI_API_KEY` configurada no arquivo `.env` na raiz do projeto
- `HF_TOKEN` configurada no arquivo `.env` na raiz do projeto

## Nota sobre HuggingFace

Este notebook utiliza o `InferenceClient` da HuggingFace, que faz chamadas à **API serverless** (Inference API).
Isso significa que os modelos rodam nos servidores da HuggingFace — **não é necessário GPU local nem download de modelos**.
É diferente do `pipeline` do Transformers, que baixa e executa o modelo localmente.
O `InferenceClient` é ideal para prototipagem rápida e funciona em qualquer máquina com acesso à internet.

In [1]:
import os
import json
import re
from datetime import datetime

import pandas as pd
import gradio as gr
from openai import OpenAI
from huggingface_hub import InferenceClient
from dotenv import load_dotenv

load_dotenv(override=True)

openai_key = os.getenv("OPENAI_API_KEY")
hf_token = os.getenv("HF_TOKEN")

if openai_key:
    print("✅ OPENAI_API_KEY detectada")
else:
    print("⚠️ OPENAI_API_KEY não encontrada — modelos OpenAI não funcionarão")

if hf_token:
    print("✅ HF_TOKEN detectada")
else:
    print("⚠️ HF_TOKEN não encontrada — modelos HuggingFace não funcionarão")

✅ OPENAI_API_KEY detectada
✅ HF_TOKEN detectada


In [9]:
# Configuração de provedores e modelos disponíveis

PROVEDORES = {
    "OpenAI": {
        "gpt-4o-mini": "gpt-4o-mini",
        "gpt-4o": "gpt-4o",
    },
    "HuggingFace": {
        "Llama 3.1 8B": "meta-llama/Llama-3.1-8B-Instruct",
        "Qwen 2.5 72B": "Qwen/Qwen2.5-72B-Instruct",
    },
}

# Templates pré-definidos com schema e descrição

TEMPLATES = {
    "Perfis de Funcionários": {
        "descricao": "Dataset de funcionários de uma empresa de tecnologia brasileira",
        "schema": {
            "nome": "nome completo brasileiro",
            "idade": "número entre 22 e 60",
            "cargo": "cargo na área de tecnologia",
            "departamento": "departamento da empresa",
            "salario": "salário mensal em reais (R$), número inteiro",
            "cidade": "cidade brasileira",
            "anos_experiencia": "número entre 0 e 35",
        },
    },
    "Vendas de Sorvete": {
        "descricao": "Registros de vendas de uma sorveteria artesanal",
        "schema": {
            "data": "data no formato YYYY-MM-DD, entre 2023-01-01 e 2024-12-31",
            "sabor": "sabor de sorvete popular no Brasil",
            "quantidade": "número de unidades vendidas (1–50)",
            "preco_unitario": "preço em reais (R$), entre 8.00 e 25.00",
            "temperatura_celsius": "temperatura do dia em °C (15–40)",
            "dia_semana": "nome do dia da semana em português",
        },
    },
    "Avaliações de Produtos": {
        "descricao": "Avaliações de clientes em um e-commerce brasileiro",
        "schema": {
            "produto": "nome do produto eletrônico",
            "categoria": "categoria do produto",
            "nota": "nota de 1 a 5 (inteiro)",
            "comentario": "comentário curto em português (1–2 frases)",
            "data_avaliacao": "data no formato YYYY-MM-DD",
            "verificado": "true ou false (compra verificada)",
        },
    },
    "Pacientes de Clínica": {
        "descricao": "Registros fictícios de pacientes de uma clínica médica",
        "schema": {
            "nome": "nome completo brasileiro",
            "idade": "número entre 1 e 90",
            "sexo": "M ou F",
            "tipo_sanguineo": "tipo sanguíneo (A+, A-, B+, B-, AB+, AB-, O+, O-)",
            "diagnostico": "condição médica comum",
            "data_consulta": "data no formato YYYY-MM-DD",
            "convenio": "nome de convênio médico brasileiro ou Particular",
        },
    },
}


def format_schema_text(schema: dict) -> str:
    """Converte um dicionário de schema em texto legível para o prompt."""
    lines = [f'- "{campo}": {descricao}' for campo, descricao in schema.items()]
    return "\n".join(lines)


print(f"Provedores configurados: {list(PROVEDORES.keys())}")
print(f"Templates disponíveis: {list(TEMPLATES.keys())}")

Provedores configurados: ['OpenAI', 'HuggingFace']
Templates disponíveis: ['Perfis de Funcionários', 'Vendas de Sorvete', 'Avaliações de Produtos', 'Pacientes de Clínica']


In [4]:
# Construção de prompts

SYSTEM_PROMPT = """Você é um gerador de dados sintéticos de alta qualidade.
Sua tarefa é gerar dados realistas e variados no formato JSON.

Regras obrigatórias:
- Retorne APENAS um JSON array válido, sem nenhum texto antes ou depois
- Não inclua markdown, explicações ou comentários
- Cada item do array deve ser um objeto JSON com os campos solicitados
- Os dados devem ser variados e realistas
- Use contexto brasileiro quando aplicável (nomes, cidades, moeda R$)"""


def build_user_prompt(template_name: str, custom_description: str, num_records: int) -> str:
    """Monta o prompt do usuário com base no template ou descrição personalizada."""
    if template_name == "Personalizado":
        return (
            f"Gere exatamente {num_records} registros no formato JSON array "
            f"com base na seguinte descrição:\n\n{custom_description}\n\n"
            f"Defina os campos mais apropriados para esse tipo de dado. "
            f"Retorne apenas o JSON array."
        )

    template = TEMPLATES[template_name]
    schema_text = format_schema_text(template["schema"])
    return (
        f"Gere exatamente {num_records} registros no formato JSON array.\n\n"
        f"Contexto: {template['descricao']}\n\n"
        f"Cada objeto deve ter exatamente estes campos:\n{schema_text}\n\n"
        f"Retorne apenas o JSON array."
    )


# Testar a construção de um prompt
exemplo = build_user_prompt("Vendas de Sorvete", "", 3)
print("Exemplo de prompt:\n")
print(exemplo)

Exemplo de prompt:

Gere exatamente 3 registros no formato JSON array.

Contexto: Registros de vendas de uma sorveteria artesanal

Cada objeto deve ter exatamente estes campos:
- "data": data no formato YYYY-MM-DD, entre 2023-01-01 e 2024-12-31
- "sabor": sabor de sorvete popular no Brasil
- "quantidade": número de unidades vendidas (1–50)
- "preco_unitario": preço em reais (R$), entre 8.00 e 25.00
- "temperatura_celsius": temperatura do dia em °C (15–40)
- "dia_semana": nome do dia da semana em português

Retorne apenas o JSON array.


In [5]:
# Funções de chamada aos modelos

def call_openai(model_id: str, user_prompt: str, temperature: float) -> str:
    """Chama a API da OpenAI e retorna o texto da resposta."""
    client = OpenAI()
    response = client.chat.completions.create(
        model=model_id,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content


def call_huggingface(model_id: str, user_prompt: str, temperature: float) -> str:
    """Chama a API serverless da HuggingFace via InferenceClient."""
    client = InferenceClient(model=model_id, token=hf_token)
    response = client.chat_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=max(temperature, 0.01),
        max_tokens=4096,
    )
    return response.choices[0].message.content


def call_model(provider: str, model_name: str, user_prompt: str, temperature: float) -> str:
    """Roteador: escolhe a função correta com base no provedor."""
    model_id = PROVEDORES[provider][model_name]
    if provider == "OpenAI":
        return call_openai(model_id, user_prompt, temperature)
    elif provider == "HuggingFace":
        return call_huggingface(model_id, user_prompt, temperature)
    else:
        raise ValueError(f"Provedor desconhecido: {provider}")


print("Funções de chamada aos modelos configuradas.")

Funções de chamada aos modelos configuradas.


In [6]:
# Parsing de JSON e geração de dados

def extract_json_from_response(response_text: str) -> list[dict]:
    """Extrai uma lista de dicionários JSON da resposta do modelo.
    
    Usa 3 estratégias de fallback:
    1. Parse direto como JSON array
    2. Extração por colchetes (encontra o primeiro [...] na resposta)
    3. JSONL por linha (cada linha que começa com '{' é um objeto)
    """
    text = response_text.strip()

    # Estratégia 1: JSON direto
    try:
        data = json.loads(text)
        if isinstance(data, list):
            return data
    except json.JSONDecodeError:
        pass

    # Estratégia 2: extrair conteúdo entre colchetes
    match = re.search(r"\[\s*\{.*\}\s*\]", text, re.DOTALL)
    if match:
        try:
            data = json.loads(match.group())
            if isinstance(data, list):
                return data
        except json.JSONDecodeError:
            pass

    # Estratégia 3: JSONL (uma linha por objeto)
    records = []
    for line in text.splitlines():
        line = line.strip().rstrip(",")
        if line.startswith("{"):
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    if records:
        return records

    raise ValueError("Não foi possível extrair JSON válido da resposta do modelo.")


def generate_data(
    provider: str,
    model_name: str,
    temperature: float,
    template_name: str,
    custom_description: str,
    num_records: int,
) -> tuple[pd.DataFrame, str, str | None]:
    """Função principal: gera os dados e retorna (DataFrame, status, csv_path)."""
    try:
        user_prompt = build_user_prompt(template_name, custom_description, int(num_records))
        raw_response = call_model(provider, model_name, user_prompt, temperature)
        records = extract_json_from_response(raw_response)

        df = pd.DataFrame(records)

        # Salvar CSV com encoding compatível com Excel
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        csv_filename = f"dados_sinteticos_{timestamp}.csv"
        csv_path = os.path.join(os.getcwd(), csv_filename)
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")

        status = (
            f"✅ {len(df)} registros gerados com sucesso!\n"
            f"Modelo: {provider} / {model_name}\n"
            f"Colunas: {', '.join(df.columns)}\n"
            f"Arquivo: {csv_filename}"
        )
        return df.head(20), status, csv_path

    except Exception as e:
        error_msg = f"❌ Erro ao gerar dados: {str(e)}"
        return pd.DataFrame(), error_msg, None


print("Funções de parsing e geração configuradas.")

Funções de parsing e geração configuradas.


In [7]:
# Interface Gradio

def get_models_for_provider(provider: str) -> gr.Dropdown:
    """Retorna as opções de modelo para o provedor selecionado."""
    models = list(PROVEDORES.get(provider, {}).keys())
    return gr.Dropdown(choices=models, value=models[0] if models else None)


def on_template_change(template_name: str):
    """Mostra/esconde o campo de descrição personalizada e exibe preview do schema."""
    if template_name == "Personalizado":
        return gr.Textbox(visible=True), "Modo personalizado — descreva o dataset desejado."
    schema = TEMPLATES[template_name]["schema"]
    preview = f"Schema: {', '.join(schema.keys())}"
    return gr.Textbox(visible=False), preview


template_options = list(TEMPLATES.keys()) + ["Personalizado"]
provider_options = list(PROVEDORES.keys())
default_models = list(PROVEDORES[provider_options[0]].keys())

with gr.Blocks(theme=gr.themes.Soft(), title="Gerador de Dados Sintéticos") as app:
    gr.Markdown("# Gerador de Dados Sintéticos com IA")
    gr.Markdown("Gere datasets sintéticos usando OpenAI ou HuggingFace, com templates pré-definidos ou descrição personalizada.")

    with gr.Row():
        with gr.Column(scale=1):
            provider_dd = gr.Dropdown(
                choices=provider_options,
                value=provider_options[0],
                label="Provedor",
            )
            model_dd = gr.Dropdown(
                choices=default_models,
                value=default_models[0],
                label="Modelo",
            )
            temperature_slider = gr.Slider(
                minimum=0.0, maximum=1.5, value=0.7, step=0.1,
                label="Temperatura",
            )
            template_dd = gr.Dropdown(
                choices=template_options,
                value=template_options[0],
                label="Tipo de Dados",
            )
            custom_desc = gr.Textbox(
                label="Descrição personalizada",
                placeholder="Ex: Tabela de jogos da Copa do Mundo com time1, time2, placar, estádio, cidade...",
                lines=3,
                visible=False,
            )
            schema_preview = gr.Textbox(
                label="Preview do schema",
                value=f"Schema: {', '.join(TEMPLATES[template_options[0]]['schema'].keys())}",
                interactive=False,
            )
            num_records_slider = gr.Slider(
                minimum=5, maximum=100, value=10, step=5,
                label="Número de registros",
            )
            generate_btn = gr.Button("Gerar Dados", variant="primary", size="lg")

        with gr.Column(scale=2):
            status_box = gr.Textbox(label="Status", interactive=False, lines=4)
            data_table = gr.Dataframe(label="Preview dos dados (até 20 linhas)", interactive=False)
            csv_file = gr.File(label="Download CSV")

    # Eventos
    provider_dd.change(
        fn=get_models_for_provider,
        inputs=[provider_dd],
        outputs=[model_dd],
    )
    template_dd.change(
        fn=on_template_change,
        inputs=[template_dd],
        outputs=[custom_desc, schema_preview],
    )
    generate_btn.click(
        fn=generate_data,
        inputs=[provider_dd, model_dd, temperature_slider, template_dd, custom_desc, num_records_slider],
        outputs=[data_table, status_box, csv_file],
    )

print("Interface Gradio montada. Execute a próxima célula para iniciar.")

Interface Gradio montada. Execute a próxima célula para iniciar.


In [10]:
app.launch(inbrowser=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
* To create a public link, set `share=True` in `launch()`.
